# Practice: Attribute an Anonymous Work

**Computational Text Analysis Pack · Module 09 — Stylometry**

*Technique:* A full attribution study, with the checks that make it publishable

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sagaustus/adh-dh-notebooks/blob/main/07-stylometry/09g_practice_attribution.ipynb)

Data: `colonial_texts.csv` — from the `data/` folder of this pack.

---

In [ ]:
# Setup — run this first.
import os, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from collections import Counter
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform

# The data this notebook needs. It is downloaded from CourseHub automatically;
# uploading by hand is only a fallback if that server is unreachable.
NEEDED = ['colonial_texts.csv']
DATA_URL = 'https://coursehub.africandigitalhumanities.ca/ide/pack-data/text/'

def _missing():
    return [f for f in NEEDED if not os.path.exists(f)]

missing = _missing()
if missing:
    # Fetch from CourseHub first, so nobody has to upload anything.
    import urllib.request, urllib.parse
    for _f in list(missing):
        try:
            urllib.request.urlretrieve(DATA_URL + urllib.parse.quote(_f), _f)
            print(f'downloaded {_f}')
        except Exception as _e:
            print(f'could not fetch {_f}: {_e}')
    missing = _missing()

if missing:
    try:
        from google.colab import files
    except ImportError:
        raise SystemExit('Place these next to the notebook: ' + ', '.join(missing))
    # Ask again until everything has arrived. The upload widget returns as soon
    # as you close it, so picking only some of the files would otherwise fail a
    # few lines below with a confusing FileNotFoundError.
    for _ in range(4):
        print('Select ALL of these at once (ctrl-click / cmd-click to multi-select):')
        print('   ' + ', '.join(missing))
        files.upload()
        missing = _missing()
        if not missing:
            break
        print('Still needed: ' + ', '.join(missing))
    if missing:
        raise SystemExit(
            'Missing: ' + ', '.join(missing) + '. Re-run this cell and select '
            'every file listed, or upload them with the folder icon on the left.')

df_colonial = pd.read_csv('colonial_texts.csv')
NIGERIAN = ['Hausa', 'Igbo', 'Yoruba', 'Nigerian Pidgin']
# Most of this pack runs on scikit-learn and the standard library. Also
# available: nltk, spacy, gensim and networkx (modules 14-19), Keras
# (case studies), and a GPU pod reached through /ide/api/neural/ for the
# neural modules — never called directly, so no token lives in a notebook.

plt.rcParams['figure.figsize'] = (10, 5.5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.25

print(f'Loaded. df_colonial has {len(df_colonial):,} rows and {df_colonial.shape[1]} columns.')

## The question

> **Given a work whose author I have hidden, can the corpus tell me who wrote
> it — and how confident should I be?**

We hold out a work with a known author, pretend we do not know, and run a real
attribution. Because the answer is checkable, we find out not just *who* the
method names but *how often it is right*.

### How to use this notebook

A worked case, not a tutorial. Run it once for the shape of the argument, then
change the question at the top to one of your own.

In [ ]:
import re

def tokenise(text):
    """Words, lowercased, Unicode-aware.

    `\\w+` matches Unicode letters, so ọ, à, ƙ, ɗ and ɓ survive. The common
    shortcut `[a-z]+` would throw them away and quietly change every count in
    this notebook — see notebook 03b.
    """
    return re.findall(r"\w+", str(text).lower())

sample = "Àwọn ọmọ ìlú ti ṣe ìpàdé — ƙungiyar ɗalibai ta yi taro"
print("sample     :", sample)
print("tokens     :", tokenise(sample))
print("naive [a-z]:", re.findall(r"[a-z]+", sample.lower()))
print()
print("The naive pattern splits words at every accented letter.")

In [ ]:
import numpy as np
works = (df_colonial.groupby(["work_id", "title", "author"])["text"]
         .apply(" ".join).reset_index())
works["n"] = works["text"].map(lambda t: len(tokenise(t)))
works = works[works["n"] >= 8000].reset_index(drop=True)
counts = works["author"].value_counts()
works = works[works["author"].isin(counts[counts >= 3].index)].reset_index(drop=True)
print(f"{len(works)} works by {works['author'].nunique()} authors with 3+ works each")
print(works["author"].value_counts().to_string())

MYSTERY = 0                       # <- change this
truth = works.loc[MYSTERY, "author"]
print(f"\nheld out: '{works.loc[MYSTERY,'title'][:44]}'  (truth hidden: {truth})")

In [ ]:
# Feature set: most-frequent words, z-scored. Module 9a's recipe.
corpus_counts = Counter()
tokenised = [tokenise(t) for t in works["text"]]
for toks in tokenised: corpus_counts.update(toks)

def features(mfw):
    feats = [w for w, _ in corpus_counts.most_common(mfw)]
    rates = np.array([[t.count(w)/len(t) for w in feats] for t in tokenised])
    mu, sd = rates.mean(0), rates.std(0); sd[sd == 0] = 1
    return (rates - mu)/sd

def cosine_delta(Z):
    nz = Z/np.linalg.norm(Z, axis=1, keepdims=True)
    return 1 - nz @ nz.T

# Attribute across several MFW bands, and see whether the answer is stable.
print(f"{'MFW':>6}{'nearest neighbour':>26}{'2nd':>22}{'margin':>9}")
print("-" * 66)
votes = Counter()
for mfw in [200, 500, 1000, 2000, 3000]:
    D = cosine_delta(features(mfw))
    d = D[MYSTERY].copy(); d[MYSTERY] = np.inf
    order = np.argsort(d)
    a1, a2 = works.loc[order[0], "author"], works.loc[order[1], "author"]
    votes[a1] += 1
    print(f"{mfw:>6}{a1[:24]:>26}{a2[:20]:>22}{d[order[1]]-d[order[0]]:>9.3f}")
print(f"\nvote across bands: {dict(votes)}")
print(f"truth            : {truth}")
print("CORRECT" if votes.most_common(1)[0][0] == truth else "WRONG")

## The check that makes it publishable

One attribution is an anecdote. Run leave-one-out across every work and you get
the method's actual error rate **on this corpus** — which is what a reader needs
in order to know how much your one attribution is worth.

In [ ]:
Z = features(2000)
D = cosine_delta(Z)
np.fill_diagonal(D, np.inf)
authors = works["author"].to_numpy()
nearest = D.argmin(axis=1)
correct = authors[nearest] == authors
print(f"leave-one-out accuracy on this corpus: {correct.mean():.3f} "
      f"({correct.sum()}/{len(correct)})")
print()
per = pd.DataFrame({"author": authors, "correct": correct}).groupby("author")["correct"].agg(["size","mean"])
print(per.to_string(float_format=lambda v: f"{v:.3f}"))
print()
print("An attribution to an author the method gets wrong half the time is worth")
print("very little, however confident the single result looked.")

In [ ]:
plt.figure(figsize=(9, 5))
Zl = linkage(squareform(D if np.isfinite(D).all() else np.nan_to_num(D, posinf=1.0),
                        checks=False), method="average")
dendrogram(Zl, labels=[f"{a[:16]}" for a in works["author"]],
           leaf_font_size=7, orientation="right")
plt.title("Cosine Delta, 2000 MFW"); plt.xlabel("distance"); plt.tight_layout()

## What this licenses

**Defensible:** "Using Cosine Delta over the 2,000 most frequent words, the
held-out work is nearest to [author]; the attribution is stable across MFW bands
from 200 to 3,000; and leave-one-out accuracy for this method on this corpus is
[X], with per-author rates in the table."

**Not defensible:** an attribution without the leave-one-out figure. It is the
only thing that tells a reader whether the method works *here* — and module 9f's
2025 reanalysis of the Federalist is a reminder that even the founding result of
the field is revisable.

**Watch the margin column.** A nearest neighbour that beats the runner-up by
0.002 is not an attribution; it is a coin flip with extra steps.

### 🔧 Your turn 1

Change MYSTERY to a work by the author with the worst per-author accuracy. Does the method still name them?

<details markdown="1">
<summary><b>What you should have seen</b> — click to expand</summary>

Usually not, and the failure is informative: authors with few works, or with
works spanning very different genres, have no stable profile for the method to
match against. That is a limit of the corpus rather than of Delta, and stating
it is part of the finding. In real disputed-authorship work this is precisely
why candidate sets are constructed carefully and why 'none of the above' has to
remain an available answer.

</details>